# 06 · Reglas candidatas observadas en este libro

## Objetivo
Aplicar a cada hoja la comparación de caudal ya definida y dejar explícito que ninguna regla queda aprobada.

## Alcance
El umbral sale de la celda «Limite minimo» o «Limite maximo» de la misma hoja. No se usa un porcentaje de BEP ni la mediana del artículo SensorLess.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from ingesta.leer_bes_jun import leer_libro
from reglas.candidatas_bes_jun import comparar_caudal_con_limites
hojas = leer_libro(libro)
vistos = []
for hoja in hojas:
    for regla in comparar_caudal_con_limites(hoja):
        vistos.append(regla["id"])
        print(
            f"id={regla['id']} revision={regla['estado_revision']} confianza={regla['nivel_confianza']} "
            f"pozo={hoja['pozo']} hoja={hoja['hoja']} "
            f"Q_bfpd={hoja['caudal_operacion_bfpd']} Qmin_bfpd={hoja['limite_minimo_bfpd']} "
            f"Qmax_bfpd={hoja['limite_maximo_bfpd']} resultado={regla['resultado']}"
        )
print("RC-BES-02_en_salida", "RC-BES-02" in vistos)


id=RC-BES-01 revision=pendiente confianza=bajo pozo=QF-084 hoja=QF-554H Q_bfpd=10301.0 Qmin_bfpd=2500.0 Qmax_bfpd=10000.0 resultado=por_encima_del_limite_maximo
id=RC-BES-03 revision=pendiente confianza=bajo pozo=QF-301 hoja=QF-545H Q_bfpd=8767.0 Qmin_bfpd=2500.0 Qmax_bfpd=10000.0 resultado=dentro_de_los_limites_escritos
id=RC-BES-03 revision=pendiente confianza=bajo pozo=QF-147 hoja=QF-500H Q_bfpd=7728.0 Qmin_bfpd=2500.0 Qmax_bfpd=10000.0 resultado=dentro_de_los_limites_escritos
id=RC-BES-03 revision=pendiente confianza=bajo pozo=QF-005 hoja=QF-378HST Q_bfpd=7116.0 Qmin_bfpd=2000.0 Qmax_bfpd=8000.0 resultado=dentro_de_los_limites_escritos
id=RC-BES-03 revision=pendiente confianza=bajo pozo=QF-306 hoja=QF-549HST Q_bfpd=5419.0 Qmin_bfpd=2000.0 Qmax_bfpd=8000.0 resultado=dentro_de_los_limites_escritos
RC-BES-02_en_salida False


## Evidencia
RC-BES-01 aparece una vez: QF-084, hoja QF-554H, 10301 BFPD frente a un máximo de 10000 BFPD. RC-BES-03 aparece en QF-301, QF-147, QF-005 y QF-306. `RC-BES-02_en_salida=False` indica que ninguna hoja tuvo el caudal de operación por debajo de su mínimo; la regla sigue escrita en `docs/reglas_candidatas.md`, pero esta corrida no la activa.

Todas las líneas salen con `revision=pendiente` y `confianza=bajo`. La celda «Cono de eficiencia» no se convirtió en regla: 8000 BFPD en S 8000N y 6000 BFPD en S6000N coinciden con un punto de la tabla, y la columna EFF, rotulada `%`, llega a 7.6 mientras el cociente de potencias de la hoja es otro orden de magnitud. Sin una curva de fabricante identificada, ese punto no se llama BEP.

## Consulta que esta evidencia no cierra
Hace falta que quien cargó la hoja diga si «Limite maximo» aplica al caudal de operación de QF-084 o a otra definición de caudal. Mientras eso no esté respondido, la regla no se aprueba y no sugiere cambiar la frecuencia.
